# Lector definitivo de facturas y recibos — Tesseract
**Configuración fija: alternativa V7 con cinco lecturas. CPU, sin entrenamiento.**

Resultado retrospectivo: 111/153 (72,55 %) en una muestra y 102/149 (68,46 %) en otra; combinado 213/302 (70,53 %), sobre 120 documentos diferentes. No garantiza ese porcentaje en documentos nuevos. Se eligió esta implementación usando esos resultados; el test final sigue pendiente.

Ejecutar de arriba abajo en Colab CPU. La sección 5 permite usar imágenes/PDF propios. La sección 6 evalúa una única configuración congelada sobre test oficial (50 documentos por base), sin búsqueda de modelos. Desactiva `EVALUAR_TEST_FINAL` si solo quieres leer tus documentos. La evaluación puede tardar por las cinco lecturas de cada imagen.

No todos los campos tienen etiquetas en las bases. Un total derivado de subtotal queda marcado para revisión. El ZIP final incluye código, configuración, resultados de test y entorno; las facturas propias se descargan por separado para evitar mezclarlas con el informe académico.


Este notebook usa `src/` del repositorio. En Colab descarga una revisión de GitHub y registra su commit; publicar primero esta estructura limpia. Localmente abrir Jupyter desde la raíz del clon y activar `USAR_CODIGO_LOCAL=True`. No contiene una copia incrustada del lector.

## 1. Configuración

In [ ]:
USAR_CODIGO_LOCAL = False
from pathlib import Path
import os, sys, json, subprocess, hashlib, shutil, time
from datetime import datetime, timezone
MODO = 'lector_definitivo'
USAR_DRIVE = False
DESCARGAR_SI_FALTAN = True
EVALUAR_TEST_FINAL = True
N_TEST_POR_BASE = 50
SEED = 42
DATOS_MANUAL = ''  # Opcional: ruta a data/raw existente en Drive o Colab.
if USAR_CODIGO_LOCAL:
    LOCAL_ROOT=Path.cwd()
    if LOCAL_ROOT.name=='notebooks':LOCAL_ROOT=LOCAL_ROOT.parent
    RAIZ=LOCAL_ROOT/'outputs/experimentos'
elif USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ=Path('/content/drive/MyDrive/Lector_Facturas_Experimentos')
else:
    RAIZ=Path('/content/Lector_Facturas_Experimentos')
RAIZ.mkdir(parents=True,exist_ok=True)
SESION=RAIZ/(MODO+'_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
SESION.mkdir()
candidatos=[Path(DATOS_MANUAL)] if DATOS_MANUAL else [
    Path('/content/Document-Scanner/data/raw'),Path('/content/Lector_Facturas_Semana3/data/raw'),RAIZ/'data/raw']
DATOS=next((p for p in candidatos if p.exists()),candidatos[-1])
DATOS.mkdir(parents=True,exist_ok=True)
print('Datos:',DATOS,'\nResultados:',SESION)
REPOSITORIO = 'Jesusl2202/Document-Scanner'
RAMA = 'main'  # Puede ser limpieza-lector-definitivo antes de integrar a main.


## 2. Instalar dependencias

In [ ]:
import urllib.request,zipfile,io
if USAR_CODIGO_LOCAL:
    CODIGO=Path.cwd()
    if CODIGO.name=='notebooks':CODIGO=CODIGO.parent
    assert (CODIGO/'src/lector_facturas/consensus.py').exists(),'Abre Jupyter dentro del repositorio limpio.'
    revision={'modo':'local','ruta':str(CODIGO)}
else:
    request=urllib.request.Request(f'https://api.github.com/repos/{REPOSITORIO}/commits/{RAMA}',headers={'User-Agent':'lector-facturas'})
    with urllib.request.urlopen(request,timeout=60) as response:commit=json.load(response)['sha']
    url=f'https://codeload.github.com/{REPOSITORIO}/zip/{commit}'
    with urllib.request.urlopen(url,timeout=120) as response:blob=response.read()
    CODIGO=SESION/'codigo';CODIGO.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for name in z.namelist():
            relative='/'.join(name.split('/')[1:])
            if not relative or name.endswith('/'):continue
            path=(CODIGO/relative).resolve()
            if not path.is_relative_to(CODIGO.resolve()):raise ValueError('Ruta no permitida')
            path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(z.read(name))
    revision={'repositorio':REPOSITORIO,'commit':commit,'rama':RAMA}
assert (CODIGO/'src/lector_facturas/consensus.py').exists(),'Publica primero el repositorio limpio o indica la rama correcta.'
(SESION/'revision_codigo.json').write_text(json.dumps(revision,indent=2),encoding='utf8')

def comando(args, cwd=None, env=None):
    r = subprocess.run(list(map(str, args)), cwd=cwd, env=env,
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-5000:]); print(r.stderr[-5000:])
        r.check_returncode()
    return r.stdout + r.stderr
if shutil.which('apt-get') and not shutil.which('tesseract'):
    print('Instalando Tesseract...')
    comando(['apt-get','update','-qq'])
    comando(['apt-get','install','-y','tesseract-ocr','tesseract-ocr-eng'])
assert shutil.which('tesseract'), 'No se encontró Tesseract. Usa Google Colab con CPU.'
print('Instalando bibliotecas...')
comando([sys.executable,'-m','pip','install','-q','-r',CODIGO/'requirements.txt'])
(SESION/'pip_freeze.txt').write_text(comando([sys.executable,'-m','pip','freeze']),encoding='utf-8')
print(comando(['tesseract','--version']).splitlines()[0])
idiomas=comando(['tesseract','--list-langs'])
print(idiomas)
assert 'eng' in {line.strip() for line in idiomas.splitlines()}, 'Falta el idioma eng de Tesseract.'

## 3. Código completo y pruebas

In [ ]:
env=os.environ.copy();env['PYTHONPATH']=str(CODIGO/'src');env['PYTHONIOENCODING']='utf-8'
log=comando([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=CODIGO,env=env)
(SESION/'pruebas.txt').write_text(log,encoding='utf-8');print(log[-1800:])
sys.path.insert(0,str(CODIGO/'src'))
# Evitar módulos de una ejecución anterior en el mismo kernel.
for key in list(sys.modules):
    if key=='lector_facturas' or key.startswith('lector_facturas.'):
        del sys.modules[key]
from lector_facturas.common import read_jsonl, write_json, write_jsonl
from lector_facturas.datasets import prepare
from lector_facturas.eda import run_eda
from lector_facturas.experiments import run, metrics, scores, bootstrap_delta
CONFIG=json.loads((CODIGO/'configs/definitivo.json').read_text())
CONFIG['seed']=SEED
assert CONFIG['extractor']=='v7'
hashes={str(p.relative_to(CODIGO)):hashlib.sha256(p.read_bytes()).hexdigest() for p in (CODIGO/'src').rglob('*.py')}
write_json(SESION/'codigo_sha256.json',hashes)
# Registrar identidad del modelo de idioma, sin copiarlo al ZIP.
paths=[]
for root in [Path('/usr/share/tesseract-ocr'),Path('/usr/share/tessdata')]:
    if root.exists():paths.extend(root.rglob('eng.traineddata'))
write_json(SESION/'modelos_idioma.json',{str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths})

## 4. Fijar el modelo preentrenado y la configuración

In [ ]:
from lector_facturas.frozen_runtime import ensure_model,final_sample,review_notes
model_info=ensure_model(RAIZ/'modelos_preentrenados'/'eng_definitivo')
CONFIG['ocr']['model_sha256']={model_info['path']:model_info['sha256']}
write_json(SESION/'modelo_fijado.json',model_info)
write_json(SESION/'configuracion_congelada.json',CONFIG)
print('Modelo fijo:',CONFIG['extractor'],'Lecturas:',len(CONFIG['ocr']['ensemble_views']))
print('Pesos verificados:',model_info['sha256'])


## 5. Leer una imagen o PDF propio
Indica `DOCUMENTO` o activa `SUBIR_DOCUMENTO` para elegirlo desde tu equipo en Colab. Cada página se procesa por separado. JSON conserva texto, cajas y evidencias; CSV conserva campos y alertas. Revisar los valores antes de utilizarlos. No se atribuye un porcentaje de exactitud a un documento sin referencia humana.


In [ ]:
DOCUMENTO = ''  # Ejemplo: '/content/factura.pdf'
SUBIR_DOCUMENTO = False
from lector_facturas.cli import predict
from lector_facturas.ocr import render_pdf
import pandas as pd
from IPython.display import display,Markdown
if SUBIR_DOCUMENTO:
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('Selecciona exactamente un archivo')
    DOCUMENTO=str(Path(next(iter(uploaded))).resolve())
if DOCUMENTO:
    source=Path(DOCUMENTO);assert source.is_file(),'No se encontró el archivo'
    destino=SESION/'documento_privado';destino.mkdir(exist_ok=True)
    pages=render_pdf(source,destino/'paginas') if source.suffix.lower()=='.pdf' else [source]
    results=[]
    for page_number,page in enumerate(pages,1):
        try:
            p=predict(page,'tesseract',CONFIG,None);p['revision']=review_notes(p)
        except Exception as exc:p={'status':'error','error':str(exc),'fields':{},'revision':['No se pudo procesar']}
        p['page']=page_number;results.append(p)
    write_json(destino/'resultado.json',{'source':str(source),'pages':results,'modelo':model_info})
    tabla=pd.DataFrame([dict(page=p['page'],status=p['status'],**p['fields'],revision='; '.join(p['revision'])) for p in results])
    tabla.to_csv(destino/'campos.csv',index=False,encoding='utf-8-sig');display(tabla)
    print('Archivos:',destino)
    try:
        from google.colab import files
        files.download(str(destino/'resultado.json'));files.download(str(destino/'campos.csv'))
    except ImportError:pass
else:print('Sin documento propio. Puedes continuar con la evaluación final.')


## 6. Evaluación final en test reservado
Se descargan las bases si faltan. Se excluyen imágenes vistas y duplicados por hash. La selección depende de semilla e identidad, no de etiquetas ni dificultad. 50 documentos por base por defecto; la configuración queda fija antes de ver resultados. No confundir este test con las muestras usadas para seleccionar la versión.


In [ ]:
if EVALUAR_TEST_FINAL:
    def localizar():
        s=next((p.parent.parent for p in DATOS.rglob('entities') if p.is_dir() and p.parent.name=='train'
                and (p.parent/'img').exists() and (p.parent/'box').exists()),None)
        w=next((p.parent for p in DATOS.rglob('class_list.txt') if (p.parent/'train.txt').exists() and (p.parent/'test.txt').exists()),None)
        return s,w
    SROIE,WILD=localizar()
    for name,path in [('sroie',SROIE),('wildreceipt',WILD)]:
        if path is None:
            assert DESCARGAR_SI_FALTAN,f'Falta {name}; configura DATOS_MANUAL.'
            print('Descargando',name,flush=True)
            print(comando([sys.executable,CODIGO/'scripts/download_data.py',name,'--out',DATOS]))
    SROIE,WILD=localizar()
    assert SROIE and WILD,'No se encontraron ambas bases.'
    rows=prepare(SROIE,WILD,SESION/'preparados',CONFIG)
    eda=run_eda(rows,SESION/'eda')
    import pandas as pd
    from IPython.display import display, Markdown
    display(pd.DataFrame([eda['split_counts']]))
    display(pd.read_csv(SESION/'eda/coverage.csv'))
else:print('Evaluación final desactivada.')

In [ ]:
if EVALUAR_TEST_FINAL:
    history=json.loads((CODIGO/'configs/documentos_revisados.json').read_text())
    sample=final_sample(rows,N_TEST_POR_BASE,set(history),set(history.values()),SEED)
    write_jsonl(SESION/'muestra_test_congelada.jsonl',sample)
    write_json(SESION/'plan_test.json',{'semilla':SEED,'documentos':len(sample),'por_base':N_TEST_POR_BASE,
        'firma':hashlib.sha256(json.dumps(sorted((r['id'],r['sha256']) for r in sample)).encode()).hexdigest(),
        'nota':'Test oficial, configuración fija; no volver a ajustar con estos resultados.'})
    FINAL_RUN=run(sample,CONFIG,SESION/'test_final',SESION/'cache_ocr')
    result=json.loads((FINAL_RUN/'evaluation/summary.json').read_text())
    accuracy=result['micro_exact_match'];processing=result['execution_success_rate']
    cumplimiento={'exactitud':accuracy,'procesamiento':processing,'cumple_exactitud_70':accuracy>=.70,
        'cumple_procesamiento_90':processing>=.90,'cumple_ambas_metas':accuracy>=.70 and processing>=.90,
        'documentos':len(sample),'campos_evaluados':result['evaluated_field_instances']}
    write_json(SESION/'cumplimiento_meta.json',cumplimiento)
    display(pd.DataFrame([cumplimiento]));display(metrics(FINAL_RUN).query('support > 0'))
    print('Mediana sin caché:',result['median_seconds_uncached'],'segundos')
    print('Meta cumplida en esta muestra.' if cumplimiento['cumple_ambas_metas'] else 'La meta no se alcanzó en esta muestra; reportar el resultado sin cambiar la selección.')


## 7. Guardar código y resultados

In [ ]:
import zipfile
report=['# Lector definitivo — resultado de ejecución','',
        'Implementación fija: alternativa V7, cinco lecturas Tesseract.',
        'Resultado retrospectivo de selección: 213/302 = 70,53 %. No es el test final.',
        'Identificador fiscal y número de comprobante sin evaluación suficiente en estas bases.','']
if EVALUAR_TEST_FINAL:report+=['## Test final',json.dumps(cumplimiento,ensure_ascii=False,indent=2)]
else:report+=['Test final no ejecutado. No se acredita cumplimiento en documentos nuevos.']
(SESION/'resumen_final.md').write_text('\n'.join(report),encoding='utf8')
ARCHIVO=SESION.parent/(SESION.name+'_resultados.zip')
with zipfile.ZipFile(ARCHIVO,'w',zipfile.ZIP_DEFLATED) as z:
    for p in SESION.rglob('*'):
        if p.is_file() and not any(part in ('cache_ocr','documento_privado','__pycache__') for part in p.parts):z.write(p,p.relative_to(SESION))
if USAR_CODIGO_LOCAL:
    with zipfile.ZipFile(ARCHIVO,'a',zipfile.ZIP_DEFLATED) as z:
        for folder in ('src','scripts','configs','tests','docs'):
            for p in (CODIGO/folder).rglob('*'):
                if p.is_file() and '__pycache__' not in p.parts:z.write(p,Path('codigo')/p.relative_to(CODIGO))
        for name in ('README.md','requirements.txt','pyproject.toml'):
            z.write(CODIGO/name,Path('codigo')/name)
print('ZIP final:',ARCHIVO)
try:
    from google.colab import files
    files.download(str(ARCHIVO))
except ImportError:pass
